### Contents
- DataSets
- CDC - SCD1 & SCD2
- Append
- Expectation- To check DQ
- Medallion Architecture
---

In [0]:
# importing Delta Live Tables
import dlt

In [0]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, lit, expr
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, LongType

In [0]:
schema = StructType([
    StructField("circuitId", IntegerType(), True),
    StructField('circuitRef', StringType(), True),
    StructField('name', StringType(), True),
    StructField('location', StringType(), True),
    StructField('country', StringType(), True),
    StructField('lat', StringType(), True),
    StructField('lng', StringType(), True),
    StructField('alt', StringType(), True),
    StructField('url', StringType(), True)
])

In [0]:
cloud_file_options = {
    'cloudFiles.format':'csv',
    'header':True
}

In [0]:
#create a bronze table from ADLS
@dlt.table(
    name = 'brz_load_circuit'
)
def bronze_load():
    df = spark.readStream.format("cloudFiles").options(**cloud_file_options).schema(schema).load('/mnt/azuredatastorageacc1425/raw/circuits.csv')
    df = df.withColumn('file_process_date', F.date_format(F.current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
    return df

In [0]:
display(`brz_load_circuit`)

In [0]:
#Expectations
#definig checks
checks={}
checks['validate circuitId col for null values'] = "(circuitId IS not NULL)"
checks['validate name col for null values'] = "(name is not NULL)"
dq_rules = "({0})".format(" AND ".join(checks.values()))
print(dq_rules)

In [0]:
@dlt.table(
    name= "stag_sliver_load_circuit"
)
@dlt.expect_all(checks)
def stag_silver_table():
    df = dlt.readStream("brz_load_circuit")
    df = df.withColumn("dq_check", F.expr(dq_rules)).filter("dq_check = true ")
    return df

In [0]:
#create new dlt table 
@dlt.create_streaming_table(
    name='silver_load_circuit'
)
#add records to the above table
@dlt.apply_changes(
    target = 'silver_load_circuit',
    source = 'stag_sliver_load_circuit',
    keys = ['circuitId'],
    stored_as_scd_type = '1',
    sequence_by = 'file_process_date'
)

In [0]:
#create table for error records
@dlt.table(
    name = ' err_silver_load_circuit'
)
@dlt.expect_all(checks)
def err_silver_load_circuit():
    df = dlt.readStream("brz_load_circuit")
    df = df.withColumn("dq_check", F.expr(dq_rules)).filter("dq_check = false ")
    return df